# Attention (Lag 10)

**Dataset**: Dow Jones Industrial Index

## Description
This notebook implements a **standard LSTM** (Hochreiter & Schmidhuber, 1997) for time series
forecasting. Uses identical hyperparameters and training protocol as LSTM-SNP for fair comparison.

- Hidden units: 8
- Single layer
- No dropout
- Stateful: hidden and cell states maintained across sequence steps

## Theory: Attention

### LSTM Cell
Standard LSTM with forget, input, cell, and output gates.
Uses `nn.LSTM` from PyTorch with `batch_first=True`.

### Output Layer
Simple linear layer: `nn.Linear(hidden_dim, 1)` maps the last hidden state to a scalar prediction.

## Model Architecture & Implementation

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
NumPy version: 2.5.1
Using device: cuda


### Attention (Lag 10) Cell

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionLag5LSTM(nn.Module):
    """
    Standard LSTM (Hochreiter & Schmidhuber, 1997) for time series forecasting.
    Stateful: maintains hidden and cell states across sequence steps.
    Same hyperparameters as LSTM-SNP for fair comparison.
    """
    def __init__(self, input_dim=1, hidden_dim=8, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.u = None  # (h, c) state

    def reset_states(self, batch_size, device):
        h0 = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        c0 = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        self.u = (h0, c0)

    def detach_states(self):
        if self.u is not None:
            self.u = (self.u[0].detach(), self.u[1].detach())

    def forward(self, x):
        if self.u is None or self.u[0].device != x.device:
            self.reset_states(x.size(0), x.device)

        lstm_out, self.u = self.lstm(x, self.u)
        out = self.fc(lstm_out[:, -1, :])
        return out


def build_model(input_dim, units):
    return AttentionLag5LSTM(input_dim=input_dim, hidden_dim=units, output_dim=1)

### Output Layer
Simple `nn.Linear(hidden_dim, 1)` — no fuzzy logic.

### Build Model

In [3]:
# Quick model check
model = build_model(input_dim=1, units=8)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters())}")

AttentionLag5LSTM(
  (lstm): LSTM(1, 8, batch_first=True)
  (fc): Linear(in_features=8, out_features=1, bias=True)
)
Total params: 361


## Data Pipeline — Dow Jones Industrial Index

In [4]:
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()

In [5]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (291,)
First 5 values: [3645.  3626.  3634.  3620.5 3607. ]


In [6]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [7]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=10)
# ============================================================

def timeseries_to_supervised(data, lag=10):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 10)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (290, 11)


In [8]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (207, 11), Val: (23, 11), Test: (60, 11)


In [9]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (207, 10, 1), y_train shape: (207,)


In [10]:
#Gaussian noise injection (Eq. 29–30)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only — never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.286404


In [11]:
#Evaluation function

def evaluate_on_test(model, X_test_eval, train, train_scaled, raw_values, scaler, device):
    """
    Warms up the model's recurrent state on clean training data, then runs
    single-step-ahead prediction on X_test_eval (clean or Gaussian-noise-corrupted
    input features). Test targets are always the clean, ground-truth values.
    """
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, len(X_raw), 1).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_test_eval)):
            X = X_test_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, len(X), 1).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_test_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)   # FIX: was predictions - meanV
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

## Training Loop

In [13]:
import os

CHECKPOINT_DIR = "checkpoints_DOW_JONES"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
val_X = val_scaled[:, 0:-1]
PATIENCE = 10

summary_path = _summary_path()
if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model(input_dim=1, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):
        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model(input_dim=1, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f} "
                  f"(best val RMSE: {ckpt['best_val_rmse']:.6f})")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)
        model = build_model(input_dim=1, units=8).to(device)
        # NOTE: no LSTM-SNP-style gate-bias init here -- this is a plain
        # nn.LSTM wrapper, so nn.LSTM's own default init is used.

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        run_losses = []

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        best_val_rmse = float('inf')
        best_state = None
        patience_ctr = 0

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0

            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                model.detach_states()  # FIX: was model.u = model.u.detach() -- self.u is a (h, c) tuple, not a tensor
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)

            _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
            print(f"Epoch {epoch + 1}/100 — train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                    break

        model.load_state_dict(best_state)
        model.reset_states(1, device)

        all_losses.append(run_losses)
        all_models.append(model)
        print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')

        predictions, rmse, mse, nmse = evaluate_on_test(
            model, X_test, train, train_scaled, raw_values, scaler, device
        )

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
            'best_val_rmse': best_val_rmse,
        }, run_ckpt_path)

    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\Attention\lag_10\checkpoints_DOW_JONES


===== RUN 1/60 =====
Epoch 1/100 — train loss: 0.137427  val RMSE: 159.601721
Epoch 2/100 — train loss: 0.090455  val RMSE: 159.506824
Epoch 3/100 — train loss: 0.090291  val RMSE: 159.450171
Epoch 4/100 — train loss: 0.090166  val RMSE: 159.409845
Epoch 5/100 — train loss: 0.090059  val RMSE: 159.379252
Epoch 6/100 — train loss: 0.089964  val RMSE: 159.357311
Epoch 7/100 — train loss: 0.089877  val RMSE: 159.339919
Epoch 8/100 — train loss: 0.089797  val RMSE: 159.325232
Epoch 9/100 — train loss: 0.089721  val RMSE: 159.312466
Epoch 10/100 — train loss: 0.089647  val RMSE: 159.300897
Epoch 11/100 — train loss: 0.089576  val RMSE: 159.289495
Epoch 12/100 — train loss: 0.089505  val RMSE: 159.277847
Epoch 13/100 — train loss: 0.089433  val RMSE: 159.264302
Epoch 14/100 — train loss: 0.089356  val RMSE: 159.246731
Epoch 15/100 — t

In [14]:
#Noise-robustness sweep (averaged over multiple noise draws, with explicit clean baseline)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

# Re-evaluate the clean (0% noise) baseline through evaluate_on_test, so it
# uses the exact same evaluation path as the noisy conditions below.
clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} ± {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} ± {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} ± {np.std(clean_nmse):.10f}")

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

[  no noise]  RMSE: 99.283008 ± 0.187146  |  MSE: 9857.150730 ± 37.181074  |  NMSE: 0.0456437803 ± 0.0001721679
[  0.5% noise]  RMSE: 99.283156 ± 0.187201  |  MSE: 9857.180163 ± 37.192022  |  NMSE: 0.0456439166 ± 0.0001722186
[  5.0% noise]  RMSE: 99.285141 ± 0.185399  |  MSE: 9857.574372 ± 36.836546  |  NMSE: 0.0456457420 ± 0.0001705725
[ 10.0% noise]  RMSE: 99.285035 ± 0.189062  |  MSE: 9857.556698 ± 37.559389  |  NMSE: 0.0456456602 ± 0.0001739197
[ 15.0% noise]  RMSE: 99.290595 ± 0.186775  |  MSE: 9858.662921 ± 37.108150  |  NMSE: 0.0456507826 ± 0.0001718302
